In [3]:
import yfinance as yf
import pandas as pd

tickers = ["WALMEX.MX", "AC.MX", "FEMSAUBD.MX", "BIMBOA.MX", "ALSEA.MX", "AAPL", "MSFT", "AMZN", "KO", "PG"]

# Data window: 2 years ending on the current date (September 7, 2026)
start_date = "2024-08-31"
end_date = "2026-08-31"

raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)["Close"]
raw_data.head()

[*******************   40%                       ]  4 of 10 completed

[*********************100%***********************]  10 of 10 completed


Ticker,AAPL,AC.MX,ALSEA.MX,AMZN,BIMBOA.MX,FEMSAUBD.MX,KO,MSFT,PG,WALMEX.MX
Date,,,,,,,,,,
2024-09-02,NaN,164.607071,54.153042,NaN,67.793777,183.738892,NaN,NaN,NaN,59.112640
2024-09-03,220.921936,165.154251,52.822525,176.250000,66.529579,179.069214,68.981865,403.072693,165.353546,57.841602
2024-09-04,219.017883,164.069183,52.171837,173.330002,68.507896,183.649246,68.556702,402.541046,166.661072,56.813557
2024-09-05,220.535187,166.712311,51.394890,177.889999,67.436707,181.865646,67.243385,402.039032,166.253632,56.028503
2024-09-06,218.988113,166.128052,50.462555,171.389999,66.770836,182.035950,67.215042,395.453003,166.367355,55.710743


In [8]:
print(raw_data.shape)
print(raw_data.isna().sum())

(514, 10)
Ticker
AAPL           15
AC.MX          16
ALSEA.MX       16
AMZN           15
BIMBOA.MX      16
FEMSAUBD.MX    16
KO             15
MSFT           15
PG             15
WALMEX.MX      16
dtype: int64


In [10]:
fondo = raw_data.ffill().dropna()
fondo.index = pd.to_datetime(fondo.index)
fondo.to_csv("fondo.csv")
print("Saved fondo.csv with shape:", fondo.shape)
print(fondo.isna().sum())

Saved fondo.csv with shape: (513, 10)
Ticker
AAPL           0
AC.MX          0
ALSEA.MX       0
AMZN           0
BIMBOA.MX      0
FEMSAUBD.MX    0
KO             0
MSFT           0
PG             0
WALMEX.MX      0
dtype: int64


In [11]:
returns = fondo.pct_change().dropna()
returns.to_csv("returns.csv")

fondo_return = returns.mean(axis=1)
fondo_return.name = "Fondo_Return"
fondo_return.head()

Date
2024-09-04   -0.000611
2024-09-05   -0.002756
2024-09-06   -0.009593
2024-09-09    0.009042
2024-09-10   -0.003491
Name: Fondo_Return, dtype: float64

In [ ]:
tech = pd.DataFrame(index=fondo_return.index)

# Lagged return (momentum, avoids leakage)
tech["Lag_Return"] = fondo_return.shift(1)

# Simple Moving Average (10-day) of the fondo's price level (equal-weighted index)
fondo_index = (1 + fondo_return).cumprod()
tech["SMA_10"] = fondo_index.rolling(window=10).mean().shift(1)

# RSI (14-day)
delta = fondo_index.diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
tech["RSI_14"] = (100 - (100 / (1 + rs))).shift(1)

tech.head(20)

In [ ]:
macro_tickers = ["^GSPC", "^MXX", "^TNX"]
macro_data = yf.download(macro_tickers, start=start_date, end=end_date, auto_adjust=True)["Close"]
macro_data = macro_data.ffill().dropna()

macro_returns = macro_data.pct_change().dropna()
macro_returns.columns = ["SP500_return", "IPC_return", "Treasury10Y_change"]
macro_returns_lagged = macro_returns.shift(1)
macro_returns_lagged.columns = [c + "_lag1" for c in macro_returns_lagged.columns]

macro_returns_lagged.head()